In [52]:
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader,TensorDataset
import  torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing


In [16]:
df = fetch_california_housing()
print(df.data.shape)
print(df.data) 
print(df.target)

(20640, 8)
[[   8.3252       41.            6.98412698 ...    2.55555556
    37.88       -122.23      ]
 [   8.3014       21.            6.23813708 ...    2.10984183
    37.86       -122.22      ]
 [   7.2574       52.            8.28813559 ...    2.80225989
    37.85       -122.24      ]
 ...
 [   1.7          17.            5.20554273 ...    2.3256351
    39.43       -121.22      ]
 [   1.8672       18.            5.32951289 ...    2.12320917
    39.43       -121.32      ]
 [   2.3886       16.            5.25471698 ...    2.61698113
    39.37       -121.24      ]]
[4.526 3.585 3.521 ... 0.923 0.847 0.894]


In [17]:
# we are solving this regression problem with Artificial Neural Network
X = df.data
y= df.target 

In [20]:
# spliting into train and test 
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [27]:
# scaling the train and test
scalar = StandardScaler()
X_train_scaled= scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

In [33]:
# converting to tensor dataset 
# train dataset
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32).view(-1,1)

# test dataset
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32).view(-1,1)

In [36]:
# maping input and target varaible 
train_data = TensorDataset(X_train_tensor,y_train_tensor)
test_data = TensorDataset(X_test_tensor,y_test_tensor)


In [40]:
# batching the all data
train = DataLoader(train_data,batch_size=32,shuffle=True)
test = DataLoader(test_data,batch_size=32,shuffle=True)

In [62]:
# building the neural network 
class ANN(nn.Module):
    def __init__(self,input_size):
        super(ANN,self).__init__() # calling the super method 
        self.model = nn.Sequential(
            # first hidden layers
            nn.Linear(input_size,10),
            nn.ReLU(),

            # output layers
            nn.Linear(10,1)
        )

    def forward(self,x):
        return self.model(x)

    def __str__(self):
        return self


In [63]:
X_train.shape[1]

8

In [65]:
# trainng and testing the model
model = ANN(X_train.shape[1])
optimizer = optim.Adam(model.parameters())
loss = nn.MSELoss()

In [69]:
epochs = 100
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for xb,yb in train:
        optimizer.zero_grad()
        output = model(xb)
        loss_value = loss(output,yb)
        loss_value.backward() #backward propagation
        optimizer.step() # updating the weight
        epoch_loss += loss_value.item()

    avg_loss = epoch_loss/ len(train)
    print(f"{epoch} Train avg loss {avg_loss}")

    

0 Train avg loss 0.347383088843767
1 Train avg loss 0.3475724062052115
2 Train avg loss 0.3466395662678767
3 Train avg loss 0.3473335996183545
4 Train avg loss 0.34712801342324695
5 Train avg loss 0.3485961351832447
6 Train avg loss 0.3472761521476877
7 Train avg loss 0.3473434161048296
8 Train avg loss 0.3478717382333075
9 Train avg loss 0.3474880314699208
10 Train avg loss 0.34751879031112953
11 Train avg loss 0.34728145928576937
12 Train avg loss 0.3465609940476427
13 Train avg loss 0.34680477409919563
14 Train avg loss 0.3471285552652769
15 Train avg loss 0.347062291259798
16 Train avg loss 0.3468333203923102
17 Train avg loss 0.34674519960328126
18 Train avg loss 0.34713790347698587
19 Train avg loss 0.34696388949257456
20 Train avg loss 0.3468721880064916
21 Train avg loss 0.3462418670253467
22 Train avg loss 0.34653299599308374
23 Train avg loss 0.346494334092898
24 Train avg loss 0.3474673974768136
25 Train avg loss 0.34714131530865217
26 Train avg loss 0.34670327186526717
27 T

In [71]:
# testing data 
model.eval()
with torch.no_grad():
    total_loss = 0
    for xb,yb in test:
        output = model(xb)
        test_loss = loss(output,yb)
        total_loss += test_loss.item()
    print(f"Loss:{total_loss/len(test)}")
        


Loss:0.3606934479271719
